# Workshop 4 — Run NVIDIA NIM on Your Own GPU

Picks up where [Workshop 3](https://github.com/torkian/nvidia-nim-workshop/blob/main/part3_guardrails.ipynb) left off. Same OpenAI-compatible code, two endpoints — NVIDIA's hosted API Catalog (what we've been using) and a NIM container you run on your own GPU.

**Heads-up about Colab:** Colab can't run the NIM Docker container itself (you can't `docker run` inside Colab). The runnable cell below verifies the hosted endpoint works, then the markdown walks you through the local setup you'd do on a machine with an NVIDIA GPU. The Python change is one line.

## Step 1 — Confirm the hosted endpoint still works (runs in Colab)

Same `ask()` we've been using since Workshop 1. If this prints a sensible answer, your Python client and API key are good.

In [ ]:
%pip install -q openai

import os, getpass
from openai import OpenAI

if not os.getenv('NVIDIA_API_KEY'):
    os.environ['NVIDIA_API_KEY'] = getpass.getpass('Paste your NVIDIA API key (starts with nvapi-): ')

HOSTED_BASE_URL = 'https://integrate.api.nvidia.com/v1'
MODEL = 'meta/llama-3.1-8b-instruct'

client = OpenAI(
    base_url=HOSTED_BASE_URL,
    api_key=os.environ['NVIDIA_API_KEY'],
)

def ask(system_prompt, user_message):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user',   'content': user_message},
        ],
        temperature=0.3,
        max_tokens=400,
    )
    return response.choices[0].message.content

print(f'Endpoint: {HOSTED_BASE_URL} (hosted)')
print('A:', ask(
    system_prompt='You are a helpful assistant. Answer in two sentences.',
    user_message='What is GPU acceleration and why does it matter for AI workloads?',
))

## Step 2 — Switch to a local NIM container (do this on a GPU box, NOT in Colab)

On a machine with an NVIDIA GPU, Linux (or WSL2), Docker, and the NVIDIA Container Toolkit installed:

**Log in to NVIDIA's container registry**
```bash
export NGC_API_KEY="nvapi-...your-key..."
echo "$NGC_API_KEY" | docker login nvcr.io --username '$oauthtoken' --password-stdin
```

**Pull and run the container**
```bash
docker run -it --rm \
  --name llama-3.1-8b-instruct \
  --runtime=nvidia \
  --gpus all \
  --shm-size=16GB \
  -e NGC_API_KEY=$NGC_API_KEY \
  -v "$HOME/.cache/nim:/opt/nim/.cache" \
  -u $(id -u) \
  -p 8000:8000 \
  nvcr.io/nim/meta/llama-3.1-8b-instruct:latest
```

When the container logs `Application startup complete`, the OpenAI-compatible endpoint is live on `http://localhost:8000/v1`.

**Verify it**
```bash
curl http://localhost:8000/v1/models
```

## Step 3 — Point the Python client at the local container

This cell shows the one-line change. **It only works on the machine running the NIM container** — running it in Colab will fail with a connection error, which is correct. (Colab can't see your laptop's `localhost`.)

In [ ]:
LOCAL_BASE_URL = 'http://localhost:8000/v1'

local_client = OpenAI(
    base_url=LOCAL_BASE_URL,                      # ← was integrate.api.nvidia.com
    api_key='not-needed-for-local-dev',           # local NIM doesn't validate the key
)

def ask_local(system_prompt, user_message):
    response = local_client.chat.completions.create(
        model=MODEL,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user',   'content': user_message},
        ],
        temperature=0.3,
        max_tokens=400,
    )
    return response.choices[0].message.content

# Run this cell ONLY on the machine hosting the NIM container.
# In Colab this will raise APIConnectionError — that's expected.
try:
    print(f'Endpoint: {LOCAL_BASE_URL} (local NIM)')
    print('A:', ask_local(
        system_prompt='You are a helpful assistant. Answer in two sentences.',
        user_message='What is GPU acceleration and why does it matter for AI workloads?',
    ))
except Exception as exc:
    print('Connection failed:', exc)
    print('\nThis is expected if you are running in Colab or do not have a local NIM container running.')
    print('See the markdown above for the docker run command.')

## What just happened

Two `OpenAI` clients, two `base_url` values, one model name, one `ask()` shape. That's the substance of Workshop 4 — the OpenAI-compatible surface NIM exposes means your application code does not care where inference is happening.

## When to use which

- **Hosted** (`integrate.api.nvidia.com`) — workshops, prototypes, demos, course projects, anything where data locality and latency are not the binding constraint.
- **Local NIM** — sensitive data that has to stay on your hardware, latency-critical workloads, large-scale inference where per-token billing becomes the bottleneck.

## Where this goes next

Workshop 5 turns the assistant into an agent — the model picks a tool, your code runs it, the result goes back to the model. The retriever from Workshop 2 and the guardrails from Workshop 3 both become things the agent can reach for.

Star the repo: https://github.com/torkian/nvidia-nim-workshop